# Use Case 1: Handwritten Digit Classification using a Basic Neural Network

This notebook introduces input layers, hidden layers, activation functions, loss, optimizers and epochs before moving to CNNs.

Each code cell is preceded by Markdown that explains what the step does, why it is needed, and which deep-learning concept is being demonstrated.


## Step 1: Import libraries

TensorFlow provides neural-network layers and model-training functions.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)


## Step 2: Configure the included digit dataset

The folder names 0 to 9 become class labels.

In [ ]:
DATASET_PATH = "../datasets/06_handwritten_digits"
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32


## Step 3: Load image files as tensors

The loader reads image files, converts them into numerical tensors and creates integer labels.

In [ ]:
DATASET_DIR = Path(DATASET_PATH)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names

print("Classes:", class_names)
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))


## Step 4: Display sample images

This confirms that the images and labels were loaded correctly.

In [ ]:
images, labels = next(iter(train_ds))

plt.figure(figsize=(12, 8))

for index in range(min(9, len(images))):
    plt.subplot(3, 3, index + 1)
    plt.imshow(images[index].numpy().astype("uint8"))
    plt.title(class_names[int(labels[index])])
    plt.axis("off")

plt.tight_layout()
plt.show()


## Step 5: Build a dense neural network

`Flatten` converts the image tensor into one long vector. Dense layers then learn relationships among pixel values.

In [ ]:
model = Sequential([
    Input(shape=(64, 64, 3)),
    Rescaling(1.0 / 255),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(10, activation="softmax"),
])

model.summary()


## Step 6: Compile the model

Adam updates weights. Sparse categorical cross-entropy is used because labels are integer class numbers.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)


## Step 7: Train the model

One epoch means one complete pass through the training dataset.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
)


## Step 8: Evaluate learning

Validation performance and learning curves help identify overfitting and underfitting.

In [ ]:
loss, accuracy = model.evaluate(val_ds)

print("Validation loss:", round(loss, 4))
print("Validation accuracy:", round(accuracy, 4))

plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()
